# Batch ETL validation

Load the localized raw Binance events and resampled Parquet output with Dask, then compare schema, cadence, daily/hourly trade totals, and a small raw-event window. Run from the repository root environment (`../../.venv`).

In [ ]:
from pathlib import Path
import json

import dask
import dask.dataframe as dd
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


## Parameters

Change `VALIDATION_HOUR` for the hourly check and `INSPECTION_START` / `INSPECTION_SECONDS` for row-level inspection. All timestamps are UTC.

In [ ]:
DATE = "2026-09-01"
VENUE = "binance"
INSTRUMENT = "BTCUSDT"
FREQUENCY = "1s"
VALIDATION_HOUR = "08"
INSPECTION_START = pd.Timestamp(f"{DATE} {VALIDATION_HOUR}:00:00", tz="UTC")
INSPECTION_SECONDS = 10

BASE = Path("tmp/validation") / DATE / VENUE
TICKS_PATH = BASE / "raw/ticks"
BOOKS_PATH = BASE / "raw/books"
BARS_PATH = BASE / "resampled" / f"frequency={FREQUENCY}"
paths = {"ticks": TICKS_PATH, "books": BOOKS_PATH, "bars": BARS_PATH}
missing = [str(path) for path in paths.values() if not path.exists()]
assert not missing, f"Missing local datasets: {missing}"
paths

## Load Parquet with Dask

Reading each dataset root enables PyArrow Hive discovery. Raw data reconstructs `venue`, `instrument`, `date`, and `hour`; resampled data reconstructs `frequency`, `date`, `hour`, and `venue`.

In [ ]:
ticks = dd.read_parquet(TICKS_PATH, engine="pyarrow")
books = dd.read_parquet(BOOKS_PATH, engine="pyarrow")
bars = dd.read_parquet(BARS_PATH, engine="pyarrow")

def typed_hour(frame, hour):
    categories = frame["hour"].cat.categories
    return int(hour) if pd.api.types.is_integer_dtype(categories.dtype) else hour

RAW_HOUR = typed_hour(ticks, VALIDATION_HOUR)
BAR_HOUR = typed_hour(bars, VALIDATION_HOUR)

overview = pd.DataFrame(
    {"partitions": [ticks.npartitions, books.npartitions, bars.npartitions],
     "columns": [len(ticks.columns), len(books.columns), len(bars.columns)]},
    index=["raw ticks", "raw books", "resampled bars"],
)
display(overview)
display(pd.DataFrame({"dtype": bars.dtypes.astype(str)}))

## Schema and partition checks

The logical Dask frame has 55 columns when loaded from `frequency=<frequency>` because all four path keys are reconstructed. Each physical Parquet file has 51 payload columns; `date`, `hour`, and `venue` exist only in its directory path.

In [ ]:
trade_columns = ["p_trade", "p_high", "p_low", "v_trade", "v_buy", "v_sell", "cnt_trade",
                 "dt_fill_mean_ms", "dt_fill_max_ms", "dt_fill_min_ms"]
book_columns = [f"{variable}_{side}_{level}" for variable in ("p", "q")
                for side in ("bid", "ask") for level in range(1, 11)]
expected_columns = ["timestamp", *trade_columns, *book_columns, "frequency", "date", "hour", "venue"]
sample_file = next(BARS_PATH.rglob("*.parquet"))
physical_columns = pq.ParquetFile(sample_file).schema_arrow.names

schema_check = pd.Series({
    "exact logical schema": set(expected_columns) == set(bars.columns),
    "logical column count is 55": len(bars.columns) == 55,
    "physical column count is 51": len(physical_columns) == 51,
    "Hive keys absent from payload": {"date", "hour", "venue"}.isdisjoint(physical_columns),
}, name="passed")
display(schema_check.to_frame(), sample_file, physical_columns)
assert schema_check.all(), schema_check[~schema_check]

## Full-day integrity and conservation

These computations scan the full local day. Float sums use tolerances because raw `quantity` is `float32` and distributed reductions can differ slightly by ordering.

In [ ]:
positive_ticks = ticks[ticks["quantity"] > 0]
raw_daily_tasks = [
    positive_ticks["quantity"].count(),
    ticks["quantity"].sum(),
    ticks[ticks["taker_side"] == "buy"]["quantity"].sum(),
    ticks[ticks["taker_side"] == "sell"]["quantity"].sum(),
]
raw_values, bar_values, row_count, unique_rows, min_timestamp, max_timestamp = dask.compute(
    raw_daily_tasks, bars[["cnt_trade", "v_trade", "v_buy", "v_sell"]].sum(),
    bars.shape[0], bars[["timestamp", "venue"]].drop_duplicates().shape[0],
    bars["timestamp"].min(), bars["timestamp"].max(),
)
raw_values = pd.Series(raw_values, index=["cnt_trade", "v_trade", "v_buy", "v_sell"], name="raw")
daily = pd.concat([raw_values, bar_values.rename("resampled")], axis=1)
daily["absolute_delta"] = daily["resampled"] - daily["raw"]
daily["relative_delta"] = daily["absolute_delta"] / daily["raw"].replace(0, np.nan)
display(daily)
display(pd.Series({"rows": row_count, "unique timestamp/venue rows": unique_rows,
                   "min timestamp": min_timestamp, "max timestamp": max_timestamp}))

In [ ]:
expected_rows = int(pd.Timedelta(days=1) / pd.Timedelta(FREQUENCY))
integrity_check = pd.Series({
    "expected row count": row_count == expected_rows,
    "timestamp/venue unique": unique_rows == row_count,
    "trade count conserved": daily.loc["cnt_trade", "absolute_delta"] == 0,
    "trade volume conserved": np.isclose(daily.loc["v_trade", "raw"], daily.loc["v_trade", "resampled"], rtol=1e-7, atol=1e-3),
    "buy volume conserved": np.isclose(daily.loc["v_buy", "raw"], daily.loc["v_buy", "resampled"], rtol=1e-7, atol=1e-3),
    "sell volume conserved": np.isclose(daily.loc["v_sell", "raw"], daily.loc["v_sell", "resampled"], rtol=1e-7, atol=1e-3),
    "buy + sell equals total": np.isclose(bar_values["v_buy"] + bar_values["v_sell"], bar_values["v_trade"], atol=1e-6),
}, name="passed")
display(integrity_check.to_frame())
assert integrity_check.all(), integrity_check[~integrity_check]

## Hourly raw-versus-resampled comparison

In [ ]:
raw_hourly_count = positive_ticks.groupby("hour", observed=True)["quantity"].count().rename("raw_cnt_trade")
raw_hourly_volume = ticks.groupby("hour", observed=True)["quantity"].sum().rename("raw_v_trade")
bar_hourly = bars.groupby("hour", observed=True)[["cnt_trade", "v_trade"]].sum()
raw_count_pd, raw_volume_pd, bar_hourly_pd = dask.compute(raw_hourly_count, raw_hourly_volume, bar_hourly)
for result in (raw_count_pd, raw_volume_pd, bar_hourly_pd):
    result.index = result.index.astype(str).str.zfill(2)
hourly = pd.concat([raw_count_pd, raw_volume_pd, bar_hourly_pd], axis=1).sort_index()
hourly["cnt_delta"] = hourly["cnt_trade"] - hourly["raw_cnt_trade"]
hourly["volume_delta"] = hourly["v_trade"] - hourly["raw_v_trade"]
display(hourly.style.format({"raw_v_trade": "{:.8f}", "v_trade": "{:.8f}", "volume_delta": "{:.10f}"}))
display(hourly.loc[[VALIDATION_HOUR]])

## Cadence, prices, and fill timing

Materialize one output hour for checks that are easier to inspect in pandas.

In [ ]:
bars_hour = bars[bars["hour"] == BAR_HOUR].compute().sort_values("timestamp").reset_index(drop=True)
timed = bars_hour.dropna(subset=["dt_fill_mean_ms"])
quality_check = pd.Series({
    "hour has expected rows": len(bars_hour) == int(pd.Timedelta(hours=1) / pd.Timedelta(FREQUENCY)),
    "cadence is exact": bars_hour["timestamp"].diff().dropna().eq(pd.Timedelta(FREQUENCY)).all(),
    "trade high >= last": bars_hour["p_high"].ge(bars_hour["p_trade"]).all(),
    "trade low <= last": bars_hour["p_low"].le(bars_hour["p_trade"]).all(),
    "best bid <= best ask": bars_hour["p_bid_1"].le(bars_hour["p_ask_1"]).all(),
    "fill min <= mean": timed["dt_fill_min_ms"].le(timed["dt_fill_mean_ms"]).all(),
    "fill mean <= max": timed["dt_fill_mean_ms"].le(timed["dt_fill_max_ms"]).all(),
}, name="passed")
display(quality_check.to_frame())
display(bars_hour.head())

## Detailed raw-event window

Show raw trades, decoded 10-level books, and output bars over the same short interval.

In [ ]:
inspection_end = INSPECTION_START + pd.Timedelta(seconds=INSPECTION_SECONDS)
start_ms, end_ms = INSPECTION_START.value // 1_000_000, inspection_end.value // 1_000_000
tick_window_dd = ticks[(ticks["exchange_ts_ms"] >= start_ms) & (ticks["exchange_ts_ms"] < end_ms)]
book_window_dd = books[(books["exchange_ts_ms"] >= start_ms) & (books["exchange_ts_ms"] < end_ms)]
bar_window_dd = bars[(bars["timestamp"] >= INSPECTION_START) & (bars["timestamp"] < inspection_end)]
tick_window, book_window, bar_window = dask.compute(tick_window_dd, book_window_dd, bar_window_dd)
tick_window = tick_window.assign(timestamp=pd.to_datetime(tick_window["exchange_ts_ms"], unit="ms", utc=True)).sort_values(["timestamp", "received_ts_ms"])
book_window = book_window.assign(timestamp=pd.to_datetime(book_window["exchange_ts_ms"], unit="ms", utc=True)).sort_values(["timestamp", "received_ts_ms"])
bar_window = bar_window.sort_values("timestamp")
print(f"Window: {INSPECTION_START} to {inspection_end} (exclusive)")
print(f"Raw trades: {len(tick_window):,}; raw book updates: {len(book_window):,}; bars: {len(bar_window):,}")
display(tick_window[["timestamp", "trade_id", "price", "quantity", "taker_side", "received_ts_ms"]])
display(bar_window[["timestamp", *trade_columns]])

In [ ]:
def decode_levels(value, side, levels=10):
    parsed = json.loads(value) if isinstance(value, str) else value
    result = {}
    for level, quote in enumerate(parsed[:levels], start=1):
        result[f"p_{side}_{level}"] = float(quote["price"])
        result[f"q_{side}_{level}"] = float(quote["quantity"])
    return result

decoded_books = pd.DataFrame([{"timestamp": row.timestamp, "sequence": row.sequence,
                               **decode_levels(row.bids, "bid"), **decode_levels(row.asks, "ask")}
                              for row in book_window.itertuples(index=False)])
display(decoded_books)
display(bar_window[["timestamp", *book_columns]])

## One-bucket aggregation comparison

Recompute one bucket directly from raw events. Book prices use the last event and quote quantities are summed. Fill timing is excluded because the first delta can depend on the preceding bucket.

In [ ]:
BUCKET_START = INSPECTION_START
bucket_end = BUCKET_START + pd.Timedelta(FREQUENCY)
bucket_ticks = tick_window[(tick_window["timestamp"] >= BUCKET_START) & (tick_window["timestamp"] < bucket_end)]
bucket_books = decoded_books[(decoded_books["timestamp"] >= BUCKET_START) & (decoded_books["timestamp"] < bucket_end)]
output_bucket = bar_window[bar_window["timestamp"] == BUCKET_START]
expected = {}
if not bucket_ticks.empty:
    expected.update(p_trade=bucket_ticks["price"].iloc[-1], p_high=bucket_ticks["price"].max(),
                    p_low=bucket_ticks["price"].min(), v_trade=bucket_ticks["quantity"].sum(),
                    v_buy=bucket_ticks.loc[bucket_ticks["taker_side"] == "buy", "quantity"].sum(),
                    v_sell=bucket_ticks.loc[bucket_ticks["taker_side"] == "sell", "quantity"].sum(),
                    cnt_trade=bucket_ticks["quantity"].gt(0).sum())
if not bucket_books.empty:
    expected.update({column: bucket_books[column].iloc[-1] for column in book_columns if column.startswith("p_")})
    expected.update({column: bucket_books[column].sum() for column in book_columns if column.startswith("q_")})
if output_bucket.empty:
    raise ValueError(f"No output row for {BUCKET_START}")
comparison = pd.DataFrame({"raw_recomputed": pd.Series(expected), "resampled": output_bucket.iloc[0]})
comparison["delta"] = comparison["resampled"] - comparison["raw_recomputed"]
display(comparison)